# Training and leave-one-domain-out evaluation

This notebook trains and compares the three methods of the thesis on the features made by the Data Preparation notebook:

| Method | Idea |
|---|---|
| **Baseline (ERM)** | learn valence/arousal from the training domains, nothing else |
| **Domain alignment (DANN)** | additionally make the hidden representation hide *which dataset* a song comes from (gradient reversal) |
| **Emotion alignment** | additionally pull songs with *similar emotions* close together and push different emotions apart, using pairs of songs **from different domains** |

**Protocol (strict domain generalization, three folds):**

| Fold (unseen test domain) | Training domains |
|---|---|
| MediaEval (DEAM + EmoMusic) | PMEmo + Memo2496 |
| Memo2496 | MediaEval + PMEmo |
| PMEmo | MediaEval + Memo2496 |

* Every choice (feature type, alignment strength, training epoch) is made **only with validation songs from the training domains**. The test domain's songs and ratings are opened only after all choices are saved and locked.
* Soundtracks is **not used at all** here (locked final test, for a later notebook).
* PMEmo results count as *exploratory*, because PMEmo was inspected in earlier experiments.

**Parts of the notebook:**

| Part | What | Uses test data? |
|---|---|---|
| 1 | Feature check: only the last MERT layer vs. a learned mix of all 13 layers (baseline, 3 seeds) | no |
| 2 | Train every method and alignment strength in every fold (5 seeds) | no |
| 3 | Lock the choices; train the "all-pairs" control for emotion alignment | no |
| 4 | Evaluate the locked models on the unseen test domain of each fold | **yes, once** |
| 5 | Tables, paired differences, bootstrap confidence intervals, review file | — |

**Run:** GPU runtime (L4 is ideal) → Run all. Every finished training run is saved immediately, so after a disconnect just Run all again; nothing finished is repeated.
**Send back:** `training_review.zip` (created by the last cell).

## Step 1 — Settings

Only the two paths at the top may need editing. Everything else is the experimental protocol: it is saved with the results, and a change starts a new, separate experiment folder (old results are never mixed with new ones).

In [1]:
from pathlib import Path

DRIVE_PROJECT = Path('/content/drive/MyDrive/Thesis - Music Emotion Recognition')
DATA_DIR = DRIVE_PROJECT / 'MER_DG_v2'            # output folder of the Data Preparation notebook

FOLDS = {                                         # test domain -> training domains
    'MediaEval': ['PMEmo', 'Memo2496'],
    'Memo2496': ['MediaEval', 'PMEmo'],
    'PMEmo': ['MediaEval', 'Memo2496'],
}
SEEDS = [1, 2, 3, 4, 5]                           # main comparison
FEATURE_CHECK_SEEDS = [1, 2, 3]                   # Part 1
VAL_FRACTION = 0.2                                # source validation share per training domain
SPLIT_SEED = 2026
NEAR_DUPLICATE_COSINE = 0.98                      # such pairs always land in the same split
MEMO_WINDOW = '15-75'                             # '15-75' (primary) or 'full' (robustness check)

TRAIN = dict(epochs=30, batch_per_domain=32, learning_rate=5e-4, weight_decay=1e-3,
             warmup_epochs=1, grad_clip=5.0, dropout=0.3, hidden=256, latent=128)
DANN_WEIGHTS = [0.03, 0.1, 0.3]                   # candidate gradient-reversal strengths
EMOTION_WEIGHTS = [0.05, 0.1, 0.2]                # candidate emotion-alignment strengths
EMOTION = dict(temperature=0.1, bandwidth=0.15)   # soft emotion target (as in an earlier exploratory notebook)
FEATURE_MODES = ['last', 'mix']                   # Part 1 candidates
SELECTION_METRIC = 'source-validation macro CCC (mean over training domains and both axes, mean over seeds)'
BOOTSTRAP_REPS = 1000

PROTOCOL = dict(version='nb17-v1', folds=FOLDS, seeds=SEEDS, feature_check_seeds=FEATURE_CHECK_SEEDS,
                val_fraction=VAL_FRACTION, split_seed=SPLIT_SEED, near_duplicate_cosine=NEAR_DUPLICATE_COSINE,
                memo_window=MEMO_WINDOW, train=TRAIN, dann_weights=DANN_WEIGHTS,
                emotion_weights=EMOTION_WEIGHTS, emotion=EMOTION, feature_modes=FEATURE_MODES,
                selection_metric=SELECTION_METRIC, bootstrap_reps=BOOTSTRAP_REPS)
print('Settings loaded.')

Settings loaded.


## Step 2 — Connect Google Drive and check the inputs from the Data Preparation notebook

No installation is needed: this notebook uses only libraries that Colab already has (PyTorch, NumPy, pandas).
The cell checks that the catalog, the feature manifest and the feature files from the Data Preparation notebook exist, and creates the experiment folder.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, sys

for needed in ['catalog/catalog.csv', 'catalog/audit_report.json', 'features/manifest.csv',
               'features/extractor.json', 'features/near_duplicates.csv']:
    if not (DATA_DIR / needed).is_file():
        raise FileNotFoundError(f'Missing {DATA_DIR / needed}. Run Notebook 16 first or fix DATA_DIR.')

def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()

def sha256_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

INPUTS = {name: sha256_file(DATA_DIR / name) for name in
          ['catalog/catalog.csv', 'features/manifest.csv', 'features/near_duplicates.csv']}
EXPERIMENT_ID = sha256_json(dict(protocol=PROTOCOL, inputs=INPUTS))[:12]
RUN_DIR = DATA_DIR / 'experiments' / f'nb17_{EXPERIMENT_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'protocol.json').write_text(json.dumps(dict(protocol=PROTOCOL, inputs=INPUTS), indent=2))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime → Change runtime type → GPU (L4 recommended), then Run all.')
DEVICE = torch.device('cuda')
print(f'✅ Inputs found. Experiment folder: {RUN_DIR}')
print(f'   GPU: {torch.cuda.get_device_name(0)} | PyTorch {torch.__version__} | Python {sys.version.split()[0]}')

Mounted at /content/drive
✅ Inputs found. Experiment folder: /content/drive/MyDrive/Thesis - Music Emotion Recognition/MER_DG_v2/experiments/nb17_cde4cd6fab60
   GPU: NVIDIA L4 | PyTorch 2.11.0+cu128 | Python 3.12.13


## Step 3 — Helper functions

Timestamped messages, a progress reporter with elapsed and **remaining time**, safe file writing, and the evaluation metrics:

* **CCC** (concordance correlation coefficient, −1…1, higher is better): agreement of predictions with ratings, penalizing both wrong ranking and wrong offset/scale. **Primary metric.**
* **Pearson r**: only the ranking/linear agreement.
* **MAE / RMSE**: average error on the 0–1 scale (×100 = points on a 0–100 scale). Not accuracy.

In [3]:
import csv, datetime, math, time, threading, zipfile, zlib, copy
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

def log(message):
    print(f'[{datetime.datetime.now():%H:%M:%S}] {message}', flush=True)

def fmt_seconds(s):
    if s is None or not math.isfinite(s):
        return '?'
    s = int(s)
    return f'{s // 3600}h {s % 3600 // 60:02d}m' if s >= 3600 else f'{s // 60}m {s % 60:02d}s'

class Progress:
    def __init__(self, total, label, every_s=30):
        self.total, self.label, self.every_s = max(int(total), 1), label, every_s
        self.done, self.start, self.last, self.lock = 0, time.monotonic(), -1e9, threading.Lock()
        log(f'▶ {label}: {total} to do')
    def step(self, n=1, note='', force=False):
        with self.lock:
            self.done += n
            now = time.monotonic()
            if force or self.done >= self.total or now - self.last >= self.every_s:
                elapsed = now - self.start
                rate = self.done / elapsed if elapsed > 0 else 0
                eta = (self.total - self.done) / rate if rate > 0 else None
                log(f'  {self.label}: {self.done}/{self.total} ({100 * self.done / self.total:.0f}%) | '
                    f'elapsed {fmt_seconds(elapsed)} | remaining ≈ {fmt_seconds(eta)} {note}')
                self.last = now

def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False,
                              default=lambda v: v.item() if hasattr(v, 'item') else str(v)))
    os.replace(tmp, path)

AXES = ('valence', 'arousal')

def axis_metrics(truth, pred):
    truth, pred = np.asarray(truth, np.float64), np.asarray(pred, np.float64)
    mt, mp = truth.mean(), pred.mean()
    vt, vp = truth.var(), pred.var()
    cov = ((truth - mt) * (pred - mp)).mean()
    err = pred - truth
    return dict(ccc=float(2 * cov / (vt + vp + (mt - mp) ** 2)) if vt + vp > 0 else float('nan'),
                pearson=float(cov / math.sqrt(vt * vp)) if vt > 0 and vp > 0 else float('nan'),
                mae=float(np.abs(err).mean()), rmse=float(np.sqrt((err ** 2).mean())), bias=float(err.mean()))

def metrics(y, pred):
    out = {}
    for i, axis in enumerate(AXES):
        for k, v in axis_metrics(y[:, i], pred[:, i]).items():
            out[f'{axis}_{k}'] = v
    for k in ('ccc', 'pearson', 'mae', 'rmse'):
        out[f'macro_{k}'] = (out[f'valence_{k}'] + out[f'arousal_{k}']) / 2
    return out

def ccc_batch(truth, pred):
    """Vectorized CCC over the last axis (used by the bootstrap)."""
    mt, mp = truth.mean(-1, keepdims=True), pred.mean(-1, keepdims=True)
    cov = ((truth - mt) * (pred - mp)).mean(-1)
    return 2 * cov / (truth.var(-1) + pred.var(-1) + (mt - mp)[..., 0] ** 2)

# Self-check of the metric code on a perfect and a shifted prediction.
_y = np.array([[0.1, 0.2], [0.5, 0.6], [0.9, 0.7]])
assert abs(metrics(_y, _y)['macro_ccc'] - 1) < 1e-12 and metrics(_y, _y + 0.1)['macro_ccc'] < 1
log('Helpers ready.')

[09:59:02] Helpers ready.


## Step 4 — Load the catalog, define labels and the source validation split

* Labels come from the catalog of the Data Preparation notebook (0–1 scale). With `MEMO_WINDOW = 'full'`, Memo2496 labels are recomputed as the mean over the whole track (robustness check only).
* **Split groups:** songs by the same artist, and pairs flagged as near-duplicates by the Data Preparation notebook, are always kept together, so the validation songs are really unseen.
* In each domain, 20 % of the groups (by number of songs) form the **source validation set** used for every choice. When a domain is the *test* domain of a fold, **all** its songs are used for testing.

In [4]:
catalog = pd.read_csv(DATA_DIR / 'catalog' / 'catalog.csv', dtype={'track_id': str}, keep_default_na=False)
catalog = catalog[catalog['dataset'] != 'Soundtracks'].reset_index(drop=True)   # locked final test: never loaded
catalog['valence'] = catalog['valence'].astype(float)
catalog['arousal'] = catalog['arousal'].astype(float)

if MEMO_WINDOW == 'full':
    with np.load(DATA_DIR / 'catalog' / 'memo2496_per_second_ratings.npz', allow_pickle=False) as f:
        uids = list(f['uids'])
        series = {u: f[f's{i}'] for i, u in enumerate(uids)}
    for i, row in catalog[catalog['dataset'] == 'Memo2496'].iterrows():
        s = series[row['uid']]
        catalog.loc[i, ['valence', 'arousal']] = (s.mean(axis=1) + 1) / 2
elif MEMO_WINDOW != '15-75':
    raise ValueError("MEMO_WINDOW must be '15-75' or 'full'")

# Union-find: artist groups + near-duplicate pairs -> split components.
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    parent[find(a)] = find(b)
for uid, group in zip(catalog['uid'], catalog['group_id']):
    union(uid, 'group:' + group)
near = pd.read_csv(DATA_DIR / 'features' / 'near_duplicates.csv')
in_catalog = set(catalog['uid'])
near = near[(near['cosine'] >= NEAR_DUPLICATE_COSINE) & near['uid_a'].isin(in_catalog) & near['uid_b'].isin(in_catalog)]
for a, b in zip(near['uid_a'], near['uid_b']):
    union(a, b)
catalog['component'] = [find(u) for u in catalog['uid']]

catalog['split'] = 'train'
for domain, part in catalog.groupby('domain'):
    sizes = part.groupby('component').size()
    order = np.random.default_rng([SPLIT_SEED, zlib.crc32(domain.encode())]).permutation(sorted(sizes.index))
    target, chosen, count = VAL_FRACTION * len(part), [], 0
    for comp in order:
        if count >= target:
            break
        chosen.append(comp)
        count += sizes[comp]
    catalog.loc[(catalog['domain'] == domain) & catalog['component'].isin(chosen), 'split'] = 'val'

overview = catalog.groupby(['domain', 'split']).size().unstack(fill_value=0)
overview['val share'] = (overview['val'] / overview.sum(axis=1)).round(3)
display(overview)
assert not set(catalog[catalog.split == 'train'].component) & set(catalog[catalog.split == 'val'].component)
catalog[['uid', 'dataset', 'domain', 'split', 'component']].to_csv(RUN_DIR / 'splits.csv', index=False)
log(f'✅ {len(catalog)} songs, {len(near)} near-duplicate pairs merged into split groups. Split saved.')

split,train,val,val share
domain,,,
MediaEval,1430,358,0.2
Memo2496,1931,483,0.2
PMEmo,603,151,0.2


[09:59:03] ✅ 4956 songs, 113 near-duplicate pairs merged into split groups. Split saved.


## Step 5 — Load all features into GPU memory

The ~5,000 small feature files are combined once into a single "pack" file on Drive (`features/pack_no_soundtracks.npz`); later sessions load only that one file, which is much faster.
Then the features are moved to the GPU (about 2 GB). For each song, the notebook remembers which 5-second windows belong to it; for Memo2496 with the 15–75 s rule only the windows starting between 15 s and 75 s are used (the whole track if it is shorter than 30 s), matching how its label was computed.

In [5]:
manifest = pd.read_csv(DATA_DIR / 'features' / 'manifest.csv')
manifest = manifest[manifest['dataset'] != 'Soundtracks'].reset_index(drop=True)
pack_path = DATA_DIR / 'features' / 'pack_no_soundtracks.npz'
pack_meta = dict(manifest_sha256=INPUTS['features/manifest.csv'])

def build_pack():
    def load_one(relpath):
        with np.load(DATA_DIR / relpath, allow_pickle=False) as f:
            return f['layers'].astype(np.float16), f['start_s'].astype(np.float32)
    progress = Progress(len(manifest), 'Reading feature files from Drive (one time only)', every_s=30)
    results = [None] * len(manifest)
    def task(i):
        results[i] = load_one(manifest.loc[i, 'feature_relpath'])
        progress.step()
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(task, range(len(manifest))))
    layers = np.concatenate([r[0] for r in results])
    starts = np.concatenate([r[1] for r in results])
    counts = np.array([len(r[1]) for r in results])
    log(f'Writing pack ({layers.nbytes / 2**30:.2f} GB) to Drive ...')
    tmp = pack_path.with_name(pack_path.name + '.tmp.npz')
    np.savez(tmp, layers=layers, starts=starts, counts=counts, uids=manifest['uid'].to_numpy(dtype=str),
             meta=np.array(json.dumps(pack_meta)))
    os.replace(tmp, pack_path)

need_build = True
if pack_path.is_file():
    with np.load(pack_path, allow_pickle=False) as f:
        need_build = json.loads(str(f['meta'])) != pack_meta
if need_build:
    build_pack()
log('Loading the feature pack ...')
with np.load(pack_path, allow_pickle=False) as f:
    PACK_LAYERS = torch.from_numpy(f['layers']).to(DEVICE)        # windows × 13 × 768, float16
    pack_starts, pack_counts, pack_uids = f['starts'], f['counts'], list(f['uids'])
N_LAYERS, WIDTH = PACK_LAYERS.shape[1], PACK_LAYERS.shape[2]
offsets = np.concatenate([[0], np.cumsum(pack_counts)[:-1]])
pack_pos = {u: i for i, u in enumerate(pack_uids)}

# Per song: the global indices of its windows (padded with -1).
window_lists = []
for row in catalog.itertuples():
    i = pack_pos[row.uid]
    idx = np.arange(offsets[i], offsets[i] + pack_counts[i])
    if row.dataset == 'Memo2496' and MEMO_WINDOW == '15-75' and float(row.duration_s) >= 30:
        starts = pack_starts[idx]                      # same rule as the label (the Data Preparation notebook)
        keep = (starts >= 15 - 1e-3) & (starts < 75 - 1e-3)
        idx = idx[keep] if keep.any() else idx
    window_lists.append(idx)
max_w = max(len(w) for w in window_lists)
WIN = torch.full((len(catalog), max_w), -1, dtype=torch.long)
for t, idx in enumerate(window_lists):
    WIN[t, :len(idx)] = torch.from_numpy(idx)
WIN = WIN.to(DEVICE)
LENGTHS = torch.tensor([len(w) for w in window_lists], device=DEVICE)
Y = torch.tensor(catalog[['valence', 'arousal']].to_numpy(), dtype=torch.float32, device=DEVICE)
DOMAIN_NAMES = sorted(catalog['domain'].unique())
log(f'✅ {PACK_LAYERS.shape[0]} windows on {torch.cuda.get_device_name(0)} '
    f'({PACK_LAYERS.numel() * 2 / 2**30:.2f} GB); {N_LAYERS} layers × {WIDTH}; up to {max_w} windows per song.')
print(catalog.assign(windows=[len(w) for w in window_lists]).groupby('dataset')['windows'].describe()[['mean', 'min', 'max']])

[09:59:03] ▶ Reading feature files from Drive (one time only): 4956 to do
[09:59:06]   Reading feature files from Drive (one time only): 1/4956 (0%) | elapsed 0m 03s | remaining ≈ 4h 19m 
[09:59:36]   Reading feature files from Drive (one time only): 755/4956 (15%) | elapsed 0m 33s | remaining ≈ 3m 06s 
[10:00:07]   Reading feature files from Drive (one time only): 1515/4956 (31%) | elapsed 1m 03s | remaining ≈ 2m 24s 
[10:00:37]   Reading feature files from Drive (one time only): 2354/4956 (47%) | elapsed 1m 33s | remaining ≈ 1m 43s 
[10:01:07]   Reading feature files from Drive (one time only): 2846/4956 (57%) | elapsed 2m 03s | remaining ≈ 1m 31s 
[10:01:37]   Reading feature files from Drive (one time only): 3509/4956 (71%) | elapsed 2m 33s | remaining ≈ 1m 03s 
[10:02:07]   Reading feature files from Drive (one time only): 4186/4956 (84%) | elapsed 3m 03s | remaining ≈ 0m 33s 
[10:02:37]   Reading feature files from Drive (one time only): 4910/4956 (99%) | elapsed 3m 33s | remaini

## Step 6 — Model, alignment losses and their self-tests

**Model** (identical for all three methods; same starting weights for the same seed):
1. *Layer use*: either only the last MERT layer (`last`) or a learned weighted mix of all 13 layers (`mix`; the weights are learned and saved, so we can later see which layers matter).
2. *Window encoder*: 768 → 256 values per 5-s window.
3. *Attentive statistics pooling*: learns which windows matter, and summarizes the song by a weighted mean and spread → 512 → 128-value song representation **z**.
4. *Head*: z → valence, arousal.

Inputs are standardized per layer with statistics from the **training songs of the current fold only**.

**Domain alignment (DANN):** a small classifier tries to tell the training domains apart from z; a *gradient reversal* layer makes the encoder work against it.
**Emotion alignment:** for each song, the similarity of z to the songs of the *other* training domain is trained to follow how similar their emotions are (Gaussian label kernel, as in an earlier exploratory notebook). The `all` scope (every other song, domain ignored) is the control.
Both auxiliary losses are switched on gradually with the standard DANN schedule.

In [6]:
from torch import nn
from torch.nn import functional as F

class SongModel(nn.Module):
    def __init__(self, feature_mode, n_layers, width, hidden, latent, dropout):
        super().__init__()
        self.feature_mode = feature_mode
        self.layer_logits = nn.Parameter(torch.zeros(n_layers)) if feature_mode == 'mix' else None
        self.window = nn.Sequential(nn.Linear(width, hidden), nn.GELU(), nn.Dropout(dropout))
        self.attention = nn.Linear(hidden, 1)
        self.project = nn.Sequential(nn.Linear(2 * hidden, latent), nn.LayerNorm(latent), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Linear(latent, 2)
    def layer_weights(self):
        return torch.softmax(self.layer_logits, 0) if self.layer_logits is not None else None
    def forward(self, x, mask):                       # x: songs × windows × layers × width
        if self.feature_mode == 'mix':
            x = (x * self.layer_weights()[None, None, :, None]).sum(2)
        else:
            x = x[:, :, -1]
        h = self.window(x)
        a = self.attention(h).squeeze(-1).masked_fill(~mask, float('-inf')).softmax(1)[..., None]
        mean = (a * h).sum(1)
        std = ((a * (h - mean[:, None]) ** 2).sum(1) + 1e-5).sqrt()
        z = self.project(torch.cat([mean, std], 1))
        return self.head(z), z

class _Reverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, coefficient):
        ctx.coefficient = coefficient
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad):
        return -ctx.coefficient * grad, None

def gradient_reverse(x, coefficient):
    return _Reverse.apply(x, coefficient)

def emotion_loss(z, y, domains, scope, temperature, bandwidth):
    n = len(y)
    eye = torch.eye(n, dtype=torch.bool, device=z.device)
    same = domains[:, None] == domains[None, :]
    mask = ~same if scope == 'cross' else ~eye
    valid = mask.any(1)
    unit = F.normalize(z, dim=1, eps=1e-8)
    mask = mask[valid]
    logits = (unit[valid] @ unit.T / temperature).masked_fill(~mask, float('-inf'))
    log_target = F.log_softmax((-torch.cdist(y[valid], y).square() / (2 * bandwidth ** 2)).masked_fill(~mask, float('-inf')), 1)
    return -(log_target.exp() * F.log_softmax(logits, 1).masked_fill(~mask, 0.)).sum(1).mean()

def ramp(progress):
    """Standard DANN schedule: 0 at the start, close to 1 after about half of training."""
    return 2 / (1 + math.exp(-10 * progress)) - 1

# Self-tests on random data
torch.manual_seed(0)
_z = torch.randn(8, 4, requires_grad=True)
gradient_reverse(_z, 0.5).sum().backward()
assert torch.allclose(_z.grad, torch.full_like(_z, -0.5))
_y, _d = torch.rand(8, 2), torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
for scope in ('cross', 'all'):
    _z.grad = None
    loss = emotion_loss(_z, _y, _d, scope, 0.1, 0.15)
    loss.backward()
    assert torch.isfinite(loss) and torch.isfinite(_z.grad).all()
_m = SongModel('mix', 13, 16, 8, 4, 0.0)
_x, _mask = torch.randn(3, 5, 13, 16), torch.tensor([[1, 1, 1, 0, 0], [1, 1, 1, 1, 1], [1, 0, 0, 0, 0]], dtype=torch.bool)
_p1, _ = _m(_x, _mask)
_x2 = _x.clone(); _x2[0, 3:] = 1e3                  # padded windows must not matter
_p2, _ = _m(_x2, _mask)
assert torch.allclose(_p1, _p2, atol=1e-5)
assert abs(ramp(0) - 0) < 1e-9 and ramp(1) > 0.99
log('✅ Model and loss self-tests passed.')

[10:02:51] ✅ Model and loss self-tests passed.


## Step 7 — Training and prediction functions

One **training run** = one fold × one method × one strength × one seed:
* Each batch contains 32 songs from **each** training domain (the smaller domain is cycled more often), so both domains have equal influence.
* AdamW optimizer, one warm-up epoch, then a cosine learning-rate decay; 30 epochs.
* After every epoch the source validation songs are predicted; the epoch with the best source-validation CCC is kept (early-stopping choice, made without test data).
* The run's history, best checkpoint and validation scores are saved to Drive immediately; finished runs are reused.

In [7]:
def song_batch(tracks, mean, std):
    tracks = torch.as_tensor(tracks, device=DEVICE)
    length = int(LENGTHS[tracks].max())
    idx = WIN[tracks, :length]
    mask = idx >= 0
    x = PACK_LAYERS[idx.clamp(min=0)].float()
    x = (x - mean) / std
    x = x * mask[..., None, None]
    return x, mask

def fold_tracks(test_domain):
    sources = FOLDS[test_domain]
    train = {d: np.flatnonzero(((catalog.domain == d) & (catalog.split == 'train')).to_numpy()) for d in sources}
    val = {d: np.flatnonzero(((catalog.domain == d) & (catalog.split == 'val')).to_numpy()) for d in sources}
    test = np.flatnonzero((catalog.domain == test_domain).to_numpy())
    return sources, train, val, test

STATS = {}
@torch.no_grad()
def fold_statistics(test_domain):
    """Per-layer mean/std over the training songs of this fold (each song weighted equally)."""
    if test_domain in STATS:
        return STATS[test_domain]
    _, train, _, _ = fold_tracks(test_domain)
    tracks = np.concatenate(list(train.values()))
    s1 = torch.zeros(N_LAYERS, WIDTH, device=DEVICE, dtype=torch.float64)
    s2 = torch.zeros_like(s1)
    for k in range(0, len(tracks), 64):
        t = torch.as_tensor(tracks[k:k + 64], device=DEVICE)
        idx = WIN[t, :int(LENGTHS[t].max())]
        mask = (idx >= 0).double()
        x = PACK_LAYERS[idx.clamp(min=0)].double()
        w = (mask / mask.sum(1, keepdim=True))[..., None, None]
        s1 += (x * w).sum(1).sum(0)
        s2 += (x ** 2 * w).sum(1).sum(0)
    mean = s1 / len(tracks)
    std = (s2 / len(tracks) - mean ** 2).clamp(min=1e-8).sqrt()
    STATS[test_domain] = (mean.float(), std.float().clamp(min=1e-3))
    return STATS[test_domain]

@torch.no_grad()
def predict(model, tracks, mean, std):
    model.eval()
    out = []
    for k in range(0, len(tracks), 64):
        x, mask = song_batch(tracks[k:k + 64], mean, std)
        out.append(model(x, mask)[0].clamp(0, 1).cpu().numpy())
    return np.concatenate(out)

def source_val_score(model, val, mean, std):
    per_domain = {d: metrics(catalog.loc[t, ['valence', 'arousal']].to_numpy(), predict(model, t, mean, std))
                  for d, t in val.items()}
    return float(np.mean([m['macro_ccc'] for m in per_domain.values()])), per_domain

def run_name(test_domain, method, feature_mode, weight, scope, seed):
    return f'{test_domain}/{method}_{feature_mode}_w{weight}_{scope}/seed{seed}'

def train_run(test_domain, method, feature_mode, weight, scope, seed):
    """method: 'baseline' | 'dann' | 'emotion'. Returns the saved result dict."""
    folder = RUN_DIR / 'runs' / run_name(test_domain, method, feature_mode, weight, scope, seed)
    if (folder / 'result.json').is_file():
        return json.loads((folder / 'result.json').read_text())
    started = time.monotonic()
    sources, train, val, _ = fold_tracks(test_domain)
    mean, std = fold_statistics(test_domain)
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = SongModel(feature_mode, N_LAYERS, WIDTH, TRAIN['hidden'], TRAIN['latent'], TRAIN['dropout']).to(DEVICE)
    params = list(model.parameters())
    discriminator = None
    if method == 'dann':
        with torch.random.fork_rng(devices=[torch.cuda.current_device()]):   # keeps the model's dropout RNG identical
            torch.manual_seed(seed + 9999)
            discriminator = nn.Sequential(nn.Linear(TRAIN['latent'], 128), nn.GELU(), nn.Linear(128, len(sources))).to(DEVICE)
        params += list(discriminator.parameters())
    optimizer = torch.optim.AdamW(params, lr=TRAIN['learning_rate'], weight_decay=TRAIN['weight_decay'])
    per = TRAIN['batch_per_domain']
    steps = math.ceil(max(len(t) for t in train.values()) / per)
    total = steps * TRAIN['epochs']
    warmup = steps * TRAIN['warmup_epochs']
    history, best = [], dict(score=-float('inf'))
    step_count = 0
    for epoch in range(TRAIN['epochs']):
        model.train()
        if discriminator is not None:
            discriminator.train()
        rng = np.random.default_rng([seed, epoch, zlib.crc32(test_domain.encode())])
        streams = []
        for d in sources:
            reps = math.ceil(steps * per / len(train[d]))
            streams.append(np.concatenate([rng.permutation(train[d]) for _ in range(reps)])[:steps * per])
        sums = dict(supervised=0., auxiliary=0., disc_correct=0, disc_total=0)
        for s in range(steps):
            lr = TRAIN['learning_rate'] * (step_count / max(warmup, 1) if step_count < warmup else
                 0.5 * (1 + math.cos(math.pi * (step_count - warmup) / max(total - warmup, 1))))
            for g in optimizer.param_groups:
                g['lr'] = lr
            tracks = np.concatenate([st[s * per:(s + 1) * per] for st in streams])
            domains = torch.arange(len(sources), device=DEVICE).repeat_interleave(per)
            x, mask = song_batch(tracks, mean, std)
            y = Y[torch.as_tensor(tracks, device=DEVICE)]
            pred, z = model(x, mask)
            supervised = F.mse_loss(pred, y)
            strength = weight * ramp(step_count / total)
            if method == 'dann':
                logits = discriminator(gradient_reverse(z, strength))
                auxiliary = F.cross_entropy(logits, domains)
                loss = supervised + auxiliary            # discriminator learns at full strength
                sums['disc_correct'] += int((logits.argmax(1) == domains).sum())
                sums['disc_total'] += len(domains)
            elif method == 'emotion':
                auxiliary = emotion_loss(z, y, domains, scope, EMOTION['temperature'], EMOTION['bandwidth'])
                loss = supervised + strength * auxiliary
            else:
                auxiliary = supervised.new_zeros(())
                loss = supervised
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Non-finite loss in {run_name(test_domain, method, feature_mode, weight, scope, seed)}')
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, TRAIN['grad_clip'])
            optimizer.step()
            sums['supervised'] += float(supervised)
            sums['auxiliary'] += float(auxiliary)
            step_count += 1
        score, per_domain = source_val_score(model, val, mean, std)
        record = dict(epoch=epoch + 1, train_mse=sums['supervised'] / steps, auxiliary=sums['auxiliary'] / steps,
                      strength=strength, val_macro_ccc=score,
                      disc_accuracy=sums['disc_correct'] / sums['disc_total'] if sums['disc_total'] else None)
        history.append(record)
        if score > best['score']:
            best = dict(score=score, epoch=epoch + 1, per_domain=per_domain,
                        state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    folder.mkdir(parents=True, exist_ok=True)
    torch.save(best['state'], folder / 'best.pt')
    model.load_state_dict(best['state'])
    weights = model.layer_weights()
    result = dict(test_domain=test_domain, sources=sources, method=method, feature_mode=feature_mode, weight=weight,
                  scope=scope, seed=seed, best_epoch=best['epoch'], val_macro_ccc=best['score'],
                  val_per_domain=best['per_domain'], seconds=time.monotonic() - started,
                  layer_weights=weights.detach().cpu().tolist() if weights is not None else None)
    atomic_json(folder / 'history.json', history)
    atomic_json(folder / 'result.json', result)
    return result

def load_model(result):
    model = SongModel(result['feature_mode'], N_LAYERS, WIDTH, TRAIN['hidden'], TRAIN['latent'], TRAIN['dropout']).to(DEVICE)
    folder = RUN_DIR / 'runs' / run_name(result['test_domain'], result['method'], result['feature_mode'],
                                          result['weight'], result['scope'], result['seed'])
    model.load_state_dict(torch.load(folder / 'best.pt', map_location=DEVICE))
    return model

def run_all(jobs, label):
    """Runs a list of (test_domain, method, feature_mode, weight, scope, seed) with an overall ETA."""
    results = []
    progress = Progress(len(jobs), label, every_s=0)
    for job in jobs:
        r = train_run(*job)
        results.append(r)
        progress.step(note=f'| {job[0]} test | {job[1]} {job[2]} w={job[3]} {job[4]} seed {job[5]} '
                           f'→ best epoch {r["best_epoch"]}, source-val CCC {r["val_macro_ccc"]:.3f} '
                           f'({r["seconds"]:.0f}s)')
    return results
log('✅ Training functions ready.')

[10:02:51] ✅ Training functions ready.


# Part 1 — Feature check: last MERT layer only vs. learned mix of all 13 layers

Baseline only, 3 seeds, all 3 folds, **source validation only**. The feature type with the higher mean source-validation CCC is used for all methods in Part 2. The decision is saved before anything else happens.

In [8]:
jobs = [(t, 'baseline', mode, 0.0, 'none', seed) for t in FOLDS for mode in FEATURE_MODES for seed in FEATURE_CHECK_SEEDS]
part1 = pd.DataFrame(run_all(jobs, 'Part 1 training runs'))
table = part1.pivot_table(index='test_domain', columns='feature_mode', values='val_macro_ccc', aggfunc='mean')
table.loc['MEAN'] = table.mean()
display(table.round(4))
decision_path = RUN_DIR / 'decision_feature_mode.json'
if decision_path.is_file():
    FEATURE_MODE = json.loads(decision_path.read_text())['feature_mode']
else:
    FEATURE_MODE = str(table.loc['MEAN'].idxmax())
    atomic_json(decision_path, dict(feature_mode=FEATURE_MODE, source_val_macro_ccc=table.loc['MEAN'].to_dict(),
                                    decided=datetime.datetime.now().isoformat(), rule='higher mean source-validation CCC'))
mix_weights = np.array([w for w in part1[part1.feature_mode == 'mix']['layer_weights']])
print('Mean learned weight per MERT layer (0 = input embedding, 12 = last):')
print(pd.Series(mix_weights.mean(0), index=range(N_LAYERS)).round(3).to_string())
log(f'✅ Part 1 decision (locked): feature mode = {FEATURE_MODE}')

[10:02:51] ▶ Part 1 training runs: 18 to do


/tmp/ipykernel_269/4182522652.py:123: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  sums['supervised'] += float(supervised)


[10:03:13]   Part 1 training runs: 1/18 (6%) | elapsed 0m 22s | remaining ≈ 6m 15s | MediaEval test | baseline last w=0.0 none seed 1 → best epoch 10, source-val CCC 0.730 (22s)
[10:03:23]   Part 1 training runs: 2/18 (11%) | elapsed 0m 31s | remaining ≈ 4m 15s | MediaEval test | baseline last w=0.0 none seed 2 → best epoch 16, source-val CCC 0.726 (10s)
[10:03:33]   Part 1 training runs: 3/18 (17%) | elapsed 0m 41s | remaining ≈ 3m 27s | MediaEval test | baseline last w=0.0 none seed 3 → best epoch 20, source-val CCC 0.717 (10s)
[10:03:43]   Part 1 training runs: 4/18 (22%) | elapsed 0m 52s | remaining ≈ 3m 02s | MediaEval test | baseline mix w=0.0 none seed 1 → best epoch 10, source-val CCC 0.764 (11s)
[10:03:53]   Part 1 training runs: 5/18 (28%) | elapsed 1m 02s | remaining ≈ 2m 42s | MediaEval test | baseline mix w=0.0 none seed 2 → best epoch 10, source-val CCC 0.768 (10s)
[10:04:04]   Part 1 training runs: 6/18 (33%) | elapsed 1m 12s | remaining ≈ 2m 25s | MediaEval test | basel

feature_mode,last,mix
test_domain,,
MediaEval,0.7243,0.7657
Memo2496,0.7739,0.8148
PMEmo,0.7206,0.7591
MEAN,0.7396,0.7799


Mean learned weight per MERT layer (0 = input embedding, 12 = last):
0     0.077
1     0.076
2     0.077
3     0.077
4     0.077
5     0.077
6     0.078
7     0.078
8     0.077
9     0.077
10    0.077
11    0.076
12    0.076
[10:06:55] ✅ Part 1 decision (locked): feature mode = mix


# Part 2 — Train all methods and alignment strengths (source data only)

For each fold: the baseline, domain alignment with 3 strengths and emotion alignment (cross-domain pairs) with 3 strengths, each with 5 seeds.
This is the longest part; the progress line shows the overall remaining time. After a disconnect, Run all continues where it stopped.

In [9]:
jobs = []
for t in FOLDS:
    for seed in SEEDS:
        jobs.append((t, 'baseline', FEATURE_MODE, 0.0, 'none', seed))
        jobs += [(t, 'dann', FEATURE_MODE, w, 'none', seed) for w in DANN_WEIGHTS]
        jobs += [(t, 'emotion', FEATURE_MODE, w, 'cross', seed) for w in EMOTION_WEIGHTS]
part2 = pd.DataFrame(run_all(jobs, 'Part 2 training runs'))
log('✅ Part 2 finished.')

[10:06:55] ▶ Part 2 training runs: 105 to do
[10:06:55]   Part 2 training runs: 1/105 (1%) | elapsed 0m 00s | remaining ≈ 0m 00s | MediaEval test | baseline mix w=0.0 none seed 1 → best epoch 10, source-val CCC 0.764 (11s)
[10:07:06]   Part 2 training runs: 2/105 (2%) | elapsed 0m 11s | remaining ≈ 10m 10s | MediaEval test | dann mix w=0.03 none seed 1 → best epoch 18, source-val CCC 0.770 (12s)
[10:07:18]   Part 2 training runs: 3/105 (3%) | elapsed 0m 23s | remaining ≈ 13m 19s | MediaEval test | dann mix w=0.1 none seed 1 → best epoch 18, source-val CCC 0.767 (12s)
[10:07:30]   Part 2 training runs: 4/105 (4%) | elapsed 0m 35s | remaining ≈ 14m 51s | MediaEval test | dann mix w=0.3 none seed 1 → best epoch 2, source-val CCC 0.733 (12s)
[10:07:44]   Part 2 training runs: 5/105 (5%) | elapsed 0m 49s | remaining ≈ 16m 30s | MediaEval test | emotion mix w=0.05 cross seed 1 → best epoch 9, source-val CCC 0.773 (14s)
[10:07:58]   Part 2 training runs: 6/105 (6%) | elapsed 1m 03s | remainin

# Part 3 — Lock the choices, then train the "all pairs" control

**Selection (source validation only):** for each fold and each alignment method, the strength with the highest mean source-validation CCC over the 5 seeds is chosen (ties → the weaker strength). The choices are written to `selection_lock.json` **before** any test-domain song is predicted.

**Control:** emotion alignment with the `all` scope (pairs from any domain, domain ignored) is trained at the **same** strength as the chosen cross-domain version. If cross-domain pairing really matters, `emotion (cross)` should beat `emotion (all)` on the unseen test domain.

In [10]:
lock_path = RUN_DIR / 'selection_lock.json'
if lock_path.is_file():
    SELECTION = json.loads(lock_path.read_text())['selection']
    log('Using the existing locked selection.')
else:
    SELECTION = {}
    for t in FOLDS:
        SELECTION[t] = {'baseline': 0.0}
        for method in ('dann', 'emotion'):
            sub = part2[(part2.test_domain == t) & (part2.method == method)]
            means = sub.groupby('weight')['val_macro_ccc'].mean()
            SELECTION[t][method] = float(means[means == means.max()].index.min())
    atomic_json(lock_path, dict(selection=SELECTION, feature_mode=FEATURE_MODE, rule=SELECTION_METRIC,
                                locked=datetime.datetime.now().isoformat(),
                                note='Written before any test-domain prediction.'))
display(pd.DataFrame(SELECTION).T)
for t in FOLDS:
    sub = part2[part2.test_domain == t].groupby(['method', 'weight'])['val_macro_ccc'].agg(['mean', 'std']).round(4)
    print(f'\nSource-validation CCC, test domain {t}:')
    print(sub.to_string())

jobs = [(t, 'emotion', FEATURE_MODE, SELECTION[t]['emotion'], 'all', seed) for t in FOLDS for seed in SEEDS]
control = pd.DataFrame(run_all(jobs, 'Part 3 control runs (emotion, all pairs)'))
log('✅ Part 3 finished: choices locked, control trained.')

,baseline,dann,emotion
MediaEval,0.0,0.03,0.2
Memo2496,0.0,0.03,0.2
PMEmo,0.0,0.10,0.2



Source-validation CCC, test domain MediaEval:
                   mean     std
method   weight                
baseline 0.00    0.7654  0.0036
dann     0.03    0.7705  0.0033
         0.10    0.7615  0.0084
         0.30    0.7153  0.0220
emotion  0.05    0.7747  0.0029
         0.10    0.7768  0.0020
         0.20    0.7780  0.0025

Source-validation CCC, test domain Memo2496:
                   mean     std
method   weight                
baseline 0.00    0.8122  0.0048
dann     0.03    0.8128  0.0046
         0.10    0.8080  0.0074
         0.30    0.7795  0.0029
emotion  0.05    0.8220  0.0044
         0.10    0.8234  0.0042
         0.20    0.8248  0.0038

Source-validation CCC, test domain PMEmo:
                   mean     std
method   weight                
baseline 0.00    0.7608  0.0051
dann     0.03    0.7647  0.0043
         0.10    0.7664  0.0025
         0.30    0.7539  0.0096
emotion  0.05    0.7686  0.0043
         0.10    0.7712  0.0014
         0.20    0.7714  0.0013


# Part 4 — Evaluate on the unseen test domain (once)

Only now are the test-domain songs predicted, using the locked choices. Evaluated per fold and seed:
* the three methods with their locked strengths,
* the emotion-alignment control (`all` pairs),
* both Part 1 baselines (`last` and `mix`), as a *descriptive* check of the feature choice (the choice itself was already locked using source data).

Predictions are saved for every song so that every number can be recomputed later.

In [11]:
if not lock_path.is_file():
    raise RuntimeError('Selection is not locked; run Part 3 first.')
def evaluations_for(t):
    return [('baseline', 'baseline', FEATURE_MODE, 0.0, 'none'),
            ('domain alignment', 'dann', FEATURE_MODE, SELECTION[t]['dann'], 'none'),
            ('emotion alignment', 'emotion', FEATURE_MODE, SELECTION[t]['emotion'], 'cross'),
            ('control: emotion all-pairs', 'emotion', FEATURE_MODE, SELECTION[t]['emotion'], 'all')]
rows, prediction_rows = [], []
progress = Progress(len(FOLDS) * (4 * len(SEEDS) + len(FEATURE_MODES) * len(FEATURE_CHECK_SEEDS)),
                    'Test-domain evaluation', every_s=20)
for t in FOLDS:
    _, _, _, test = fold_tracks(t)
    mean, std = fold_statistics(t)
    truth = catalog.loc[test, ['valence', 'arousal']].to_numpy()
    plan = [(label, m, f, w, s, seed) for (label, m, f, w, s) in evaluations_for(t) for seed in SEEDS]
    plan += [(f'feature check: {mode}', 'baseline', mode, 0.0, 'none', seed)
             for mode in FEATURE_MODES for seed in FEATURE_CHECK_SEEDS]
    for label, method, mode, weight, scope, seed in plan:
        result = json.loads((RUN_DIR / 'runs' / run_name(t, method, mode, weight, scope, seed) / 'result.json').read_text())
        pred = predict(load_model(result), test, mean, std)
        rows.append(dict(test_domain=t, model=label, weight=weight, seed=seed, best_epoch=result['best_epoch'],
                         **metrics(truth, pred)))
        prediction_rows.append(pd.DataFrame(dict(test_domain=t, model=label, seed=seed, uid=catalog.loc[test, 'uid'].values,
                                                 true_valence=truth[:, 0], pred_valence=pred[:, 0],
                                                 true_arousal=truth[:, 1], pred_arousal=pred[:, 1])))
        progress.step()
test_results = pd.DataFrame(rows)
test_results.to_csv(RUN_DIR / 'test_results_per_seed.csv', index=False)
predictions = pd.concat(prediction_rows)
predictions.to_csv(RUN_DIR / 'test_predictions.csv.gz', index=False, compression='gzip')
log('✅ Part 4 finished: test results saved.')

[10:37:05] ▶ Test-domain evaluation: 78 to do
[10:37:05]   Test-domain evaluation: 1/78 (1%) | elapsed 0m 00s | remaining ≈ 0m 05s 
[10:37:09]   Test-domain evaluation: 78/78 (100%) | elapsed 0m 04s | remaining ≈ 0m 00s 
[10:37:12] ✅ Part 4 finished: test results saved.


# Part 5 — Results

1. **Main table** per fold: mean ± SD over seeds for valence/arousal CCC (primary), Pearson, MAE and RMSE (MAE/RMSE ×100 = points on a 0–100 scale).
2. **Paired comparison** with the baseline (same seeds): per-seed differences in macro CCC, and a 95 % bootstrap interval over test songs (seed-averaged predictions, 1,000 resamples). A method is called *better* only if it wins in at least 4 of 5 seeds **and** the interval is above zero.
3. **Thesis criterion**: in how many folds is emotion alignment > domain alignment > baseline?

In [12]:
main_models = ['baseline', 'domain alignment', 'emotion alignment', 'control: emotion all-pairs']
summary_rows = []
for (t, m), g in test_results.groupby(['test_domain', 'model']):
    row = dict(test_domain=t, model=m, seeds=len(g))
    for k in ('macro_ccc', 'valence_ccc', 'arousal_ccc', 'macro_pearson', 'macro_mae', 'macro_rmse'):
        row[k] = f'{g[k].mean():.3f} ± {g[k].std(ddof=1):.3f}' if len(g) > 1 else f'{g[k].mean():.3f}'
        row[k + '_mean'] = g[k].mean()
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(RUN_DIR / 'summary.csv', index=False)
for t in FOLDS:
    print(f'\n━━━ Unseen test domain: {t}  (trained on {" + ".join(FOLDS[t])}) ━━━')
    display(summary[summary.test_domain == t].set_index('model')[
        ['seeds', 'macro_ccc', 'valence_ccc', 'arousal_ccc', 'macro_pearson', 'macro_mae', 'macro_rmse']])

def paired(t, model, reference='baseline'):
    a = test_results[(test_results.test_domain == t) & (test_results.model == model)].set_index('seed')['macro_ccc']
    b = test_results[(test_results.test_domain == t) & (test_results.model == reference)].set_index('seed')['macro_ccc']
    diffs = (a - b).dropna()
    pa = predictions[(predictions.test_domain == t) & (predictions.model == model)]
    pb = predictions[(predictions.test_domain == t) & (predictions.model == reference)]
    cols = ['pred_valence', 'pred_arousal']
    ma, mb = pa.groupby('uid')[cols].mean(), pb.groupby('uid')[cols].mean()
    truth = pa.groupby('uid')[['true_valence', 'true_arousal']].first().loc[ma.index].to_numpy().T
    pred_a, pred_b = ma.to_numpy().T, mb.loc[ma.index].to_numpy().T
    rng = np.random.default_rng(zlib.crc32(f'{t}{model}{reference}'.encode()))
    idx = rng.integers(0, truth.shape[1], size=(BOOTSTRAP_REPS, truth.shape[1]))
    boot = (ccc_batch(truth[:, idx], pred_a[:, idx]) - ccc_batch(truth[:, idx], pred_b[:, idx])).mean(0)
    low, high = np.percentile(boot, [2.5, 97.5])
    wins = int((diffs > 0).sum())
    return dict(test_domain=t, comparison=f'{model} − {reference}', mean_seed_difference=diffs.mean(),
                seeds_better=f'{wins}/{len(diffs)}', ci95_low=low, ci95_high=high,
                better=bool(wins >= math.ceil(0.8 * len(diffs)) and low > 0))

paired_rows = []
for t in FOLDS:
    for m in ('domain alignment', 'emotion alignment', 'control: emotion all-pairs'):
        paired_rows.append(paired(t, m))
    paired_rows.append(paired(t, 'emotion alignment', 'domain alignment'))
    paired_rows.append(paired(t, 'emotion alignment', 'control: emotion all-pairs'))
paired_df = pd.DataFrame(paired_rows)
paired_df.to_csv(RUN_DIR / 'paired_comparisons.csv', index=False)
print('\nPaired comparisons (macro CCC; positive = first model better):')
display(paired_df.round(4))

mean_ccc = test_results.groupby(['test_domain', 'model'])['macro_ccc'].mean().unstack()
criterion = {t: bool(mean_ccc.loc[t, 'emotion alignment'] > mean_ccc.loc[t, 'domain alignment'] > mean_ccc.loc[t, 'baseline'])
             for t in FOLDS}
print('\nThesis ranking emotion > domain > baseline (mean macro CCC):',
      {t: '✓' if ok else '✗' for t, ok in criterion.items()}, f'→ {sum(criterion.values())}/3 folds')
print('Note: the ranking uses means; whether differences are reliable is shown by the paired table above.')
print('Note: PMEmo results are exploratory (PMEmo was inspected in earlier experiments).')
atomic_json(RUN_DIR / 'criterion.json', dict(criterion=criterion, mean_macro_ccc=mean_ccc.to_dict()))


━━━ Unseen test domain: MediaEval  (trained on PMEmo + Memo2496) ━━━


,seeds,macro_ccc,valence_ccc,arousal_ccc,macro_pearson,macro_mae,macro_rmse
model,,,,,,,
baseline,5,0.662 ± 0.013,0.531 ± 0.022,0.793 ± 0.008,0.682 ± 0.010,0.092 ± 0.002,0.116 ± 0.002
control: emotion all-pairs,5,0.665 ± 0.008,0.536 ± 0.016,0.794 ± 0.006,0.693 ± 0.009,0.091 ± 0.002,0.115 ± 0.003
domain alignment,5,0.663 ± 0.026,0.541 ± 0.045,0.786 ± 0.013,0.691 ± 0.028,0.093 ± 0.004,0.118 ± 0.005
emotion alignment,5,0.661 ± 0.014,0.529 ± 0.023,0.792 ± 0.009,0.685 ± 0.011,0.091 ± 0.001,0.115 ± 0.001
feature check: last,3,0.601 ± 0.009,0.459 ± 0.005,0.742 ± 0.014,0.648 ± 0.007,0.097 ± 0.001,0.122 ± 0.001
feature check: mix,3,0.666 ± 0.012,0.533 ± 0.022,0.799 ± 0.002,0.686 ± 0.009,0.091 ± 0.002,0.115 ± 0.002



━━━ Unseen test domain: Memo2496  (trained on MediaEval + PMEmo) ━━━


,seeds,macro_ccc,valence_ccc,arousal_ccc,macro_pearson,macro_mae,macro_rmse
model,,,,,,,
baseline,5,0.570 ± 0.014,0.496 ± 0.012,0.643 ± 0.020,0.661 ± 0.006,0.135 ± 0.006,0.168 ± 0.008
control: emotion all-pairs,5,0.582 ± 0.010,0.506 ± 0.018,0.658 ± 0.017,0.667 ± 0.004,0.131 ± 0.004,0.163 ± 0.005
domain alignment,5,0.574 ± 0.013,0.484 ± 0.014,0.664 ± 0.016,0.659 ± 0.009,0.135 ± 0.008,0.167 ± 0.008
emotion alignment,5,0.590 ± 0.012,0.510 ± 0.020,0.669 ± 0.015,0.668 ± 0.005,0.129 ± 0.004,0.160 ± 0.006
feature check: last,3,0.580 ± 0.011,0.483 ± 0.011,0.677 ± 0.011,0.626 ± 0.005,0.114 ± 0.002,0.143 ± 0.003
feature check: mix,3,0.566 ± 0.016,0.494 ± 0.016,0.638 ± 0.021,0.658 ± 0.007,0.136 ± 0.005,0.169 ± 0.007



━━━ Unseen test domain: PMEmo  (trained on MediaEval + Memo2496) ━━━


,seeds,macro_ccc,valence_ccc,arousal_ccc,macro_pearson,macro_mae,macro_rmse
model,,,,,,,
baseline,5,0.591 ± 0.020,0.509 ± 0.048,0.674 ± 0.024,0.699 ± 0.017,0.108 ± 0.003,0.131 ± 0.002
control: emotion all-pairs,5,0.590 ± 0.013,0.488 ± 0.024,0.691 ± 0.014,0.691 ± 0.004,0.107 ± 0.002,0.130 ± 0.002
domain alignment,5,0.582 ± 0.025,0.488 ± 0.032,0.677 ± 0.024,0.700 ± 0.023,0.108 ± 0.004,0.131 ± 0.004
emotion alignment,5,0.603 ± 0.017,0.511 ± 0.028,0.695 ± 0.021,0.698 ± 0.006,0.106 ± 0.002,0.129 ± 0.002
feature check: last,3,0.512 ± 0.019,0.431 ± 0.012,0.594 ± 0.034,0.637 ± 0.019,0.118 ± 0.003,0.144 ± 0.003
feature check: mix,3,0.588 ± 0.005,0.499 ± 0.038,0.678 ± 0.033,0.703 ± 0.006,0.110 ± 0.001,0.133 ± 0.001



Paired comparisons (macro CCC; positive = first model better):


,test_domain,comparison,mean_seed_difference,seeds_better,ci95_low,ci95_high,better
0,MediaEval,domain alignment − baseline,0.0012,2/5,-0.0015,0.0061,False
1,MediaEval,emotion alignment − baseline,-0.0014,3/5,-0.0125,-0.0010,False
2,MediaEval,control: emotion all-pairs − baseline,0.0030,3/5,-0.0070,0.0034,False
3,MediaEval,emotion alignment − domain alignment,-0.0026,2/5,-0.0150,-0.0029,False
4,MediaEval,emotion alignment − control: emotion all-pairs,-0.0045,3/5,-0.0083,-0.0022,False
5,Memo2496,domain alignment − baseline,0.0042,3/5,0.0039,0.0089,False
6,Memo2496,emotion alignment − baseline,0.0199,5/5,0.0132,0.0190,True
7,Memo2496,control: emotion all-pairs − baseline,0.0122,4/5,0.0038,0.0099,True
8,Memo2496,emotion alignment − domain alignment,0.0157,4/5,0.0063,0.0130,True
9,Memo2496,emotion alignment − control: emotion all-pairs,0.0077,5/5,0.0078,0.0105,True



Thesis ranking emotion > domain > baseline (mean macro CCC): {'MediaEval': '✗', 'Memo2496': '✓', 'PMEmo': '✗'} → 1/3 folds
Note: the ranking uses means; whether differences are reliable is shown by the paired table above.
Note: PMEmo results are exploratory (PMEmo was inspected in earlier experiments).


## Step 8 — Create the review file to send back

Packs protocol, splits, decisions, all run histories and results, test results, predictions, summary and paired comparisons into `training_review.zip` (model checkpoints are not included).

In [13]:
review = RUN_DIR / 'training_review.zip'
with zipfile.ZipFile(review, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in RUN_DIR.rglob('*'):
        if path.is_file() and path != review and path.suffix in ('.json', '.csv', '.gz'):
            z.write(path, path.relative_to(RUN_DIR).as_posix())
log(f'✅ Review file ready on Drive: {review}')
try:
    from google.colab import files
    files.download(str(review))
except Exception as error:
    print('Automatic download not available; take the file from Drive instead.', error)

[10:37:23] ✅ Review file ready on Drive: /content/drive/MyDrive/Thesis - Music Emotion Recognition/MER_DG_v2/experiments/nb17_cde4cd6fab60/training_review.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>